In [3]:
from hdfs import InsecureClient

In [4]:
!hostname -I

172.17.0.2 172.20.0.6 


In [22]:
# 경로내 목록 읽어오기
user = 'hadoop'
host = 'http://namenode:9870'
hdfs_path = '/dataset'
hdfs = InsecureClient(host,user)

show = hdfs.list(hdfs_path)
for s in show:
    print(s)

부산광역시방범용CCTV정보202512.csv
주민등록인구및세대현황202512.csv
행정구역시도성연령별취업자202606.csv
hdfs청소년정신건강20260604.csv


In [11]:
# 파일읽기
file_name = hdfs_path + "/부산광역시방범용CCTV정보202512.csv"
with hdfs.read(file_name, encoding='cp949') as reader:
    text = reader.read()
print(text[:100])

연번,구군,장비종류,관리번호,시설명칭,위도,경도
1,중구,생활방범,3250000-00001-0-1HA,0001-1-영주1동-영주1동주민센터_옆_사거리,35.110905,129.0


In [ ]:
# 워드카운트
from collections import Counter

word = text[:5000].strip().split(',')
word_counts = Counter(word)
for word, count in word_counts.items():
    print(word,count)

In [32]:
# 파일업로드
local_file = './dataset/청소년정신건강20260604.csv'
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
hdfs.upload(hdfs_file, local_file ,overwrite = True)
print("업로드완료")

업로드완료


In [35]:
# 파일확인
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
if hdfs.status(hdfs_file, strict=False)==None:
    print(f"{hdfs_file[9:]} 파일이 없습니다.")
else:
    print(hdfs.status(hdfs_file, strict=False))

hdfs청소년정신건강20260604.csv 파일이 없습니다.


In [36]:
# 파일삭제
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
if hdfs.status(hdfs_file, strict=False)==None:
    print(f"{hdfs_file[9:]} 파일이 없습니다.")
else:
    hdfs.delete(hdfs_file)
    print("삭제되었습니다.")

hdfs청소년정신건강20260604.csv 파일이 없습니다.


---
## 전체 코드 정리 (HDFS CRUD 요약)

| 기능 | 메서드 | 설명 |
|------|--------|------|
| 목록 조회 | `hdfs.list(path)` | 경로 내 파일/폴더 목록 반환 |
| 파일 읽기 | `hdfs.read(path, encoding)` | 파일 내용을 스트림으로 읽기 |
| 파일 업로드 | `hdfs.upload(hdfs_path, local_path, overwrite)` | 로컬 파일을 HDFS에 업로드 |
| 파일 존재 확인 | `hdfs.status(path, strict=False)` | 파일 메타정보 반환 (없으면 None) |
| 파일 삭제 | `hdfs.delete(path)` | HDFS 파일 삭제 |

In [ ]:
# ===== HDFS CRUD 전체 정리 코드 =====
from hdfs import InsecureClient

# 클라이언트 연결
user = 'hadoop'
host = 'http://namenode:9870'
hdfs_path = '/dataset'
hdfs = InsecureClient(host, user=user)

# 1. 목록 조회 (Read - List)
print("=" * 40)
print("[1] HDFS 디렉토리 목록 조회")
file_list = hdfs.list(hdfs_path)
for f in file_list:
    print(f"  - {f}")


In [ ]:

# 2. 파일 읽기 (Read)
print("\n[2] 파일 읽기")
file_name = hdfs_path + "/부산광역시방범용CCTV정보202512.csv"
with hdfs.read(file_name, encoding='cp949') as reader:
    text = reader.read()
print(f"  파일 크기: {len(text)} 글자")
print(f"  첫 100자: {text[:100]}")


In [ ]:

# 3. 파일 업로드 (Create)
print("\n[3] 파일 업로드")
local_file = './dataset/청소년정신건강20260604.csv'
hdfs_file = '/dataset/hdfs청소년정신건강20260604.csv'
hdfs.upload(hdfs_file, local_file, overwrite=True)
print(f"  업로드 완료: {hdfs_file}")


In [ ]:

# 4. 파일 존재 확인 (Read - Status)
print("\n[4] 파일 존재 확인")
status = hdfs.status(hdfs_file, strict=False)
if status is None:
    print(f"  {hdfs_file[9:]} 파일이 없습니다.")
else:
    print(f"  파일 존재 확인! 크기: {status['length']} bytes")


In [ ]:

# 5. 파일 삭제 (Delete)
print("\n[5] 파일 삭제")
status = hdfs.status(hdfs_file, strict=False)
if status is None:
    print(f"  {hdfs_file[9:]} 파일이 없습니다.")
else:
    hdfs.delete(hdfs_file)
    print(f"  {hdfs_file[9:]} 삭제 완료!")
print("=" * 40)

---
## 연습용 개선 코드

아래 코드는 다음 기능을 추가로 연습합니다:
- **HdfsManager 클래스**로 CRUD 기능 캡슐화
- **pandas DataFrame** 연동으로 CSV 데이터 분석
- **디렉토리 생성/삭제** (mkdir, delete recursive)
- **파일 이름 변경** (rename)
- **파일 목록 상세 정보** (status per file)
- **텍스트 파일 쓰기** (write)

In [ ]:
# ===== 연습용 개선 코드: HdfsManager 클래스 =====
from hdfs import InsecureClient
import pandas as pd
import io

class HdfsManager:
    """HDFS CRUD 작업을 캡슐화한 매니저 클래스"""

    def __init__(self, host='http://namenode:9870', user='hadoop'):
        self.hdfs = InsecureClient(host, user=user)
        print(f"HDFS 연결: {host} (user={user})")

    # --- 조회 기능 ---
    def list_dir(self, path, detail=False):
        """디렉토리 목록 조회 (detail=True면 크기/날짜 포함)"""
        files = self.hdfs.list(path, status=detail)
        if detail:
            print(f"{'파일명':<40} {'크기(bytes)':>12} {'타입'}")
            print("-" * 60)
            for name, info in files:
                print(f"{name:<40} {info['length']:>12} {info['type']}")
        else:
            for f in files:
                print(f"  - {f}")
        return files

    def exists(self, path):
        """파일/디렉토리 존재 여부 확인"""
        return self.hdfs.status(path, strict=False) is not None

    # --- 읽기 기능 ---
    def read_csv(self, hdfs_path, encoding='cp949'):
        """HDFS CSV 파일을 pandas DataFrame으로 읽기"""
        with self.hdfs.read(hdfs_path, encoding=encoding) as reader:
            text = reader.read()
        df = pd.read_csv(io.StringIO(text))
        print(f"  로드 완료: {df.shape[0]}행 x {df.shape[1]}열")
        return df

    def read_text(self, hdfs_path, encoding='utf-8', preview=200):
        """텍스트 파일 미리보기"""
        with self.hdfs.read(hdfs_path, encoding=encoding) as reader:
            text = reader.read()
        print(f"  전체 크기: {len(text)} 글자")
        print(f"  미리보기:\n{text[:preview]}")
        return text

    # --- 쓰기 기능 ---
    def write_text(self, hdfs_path, content, encoding='utf-8', overwrite=True):
        """텍스트 내용을 HDFS 파일로 직접 쓰기"""
        data = content.encode(encoding)
        with self.hdfs.write(hdfs_path, overwrite=overwrite) as writer:
            writer.write(data)
        print(f"  쓰기 완료: {hdfs_path} ({len(data)} bytes)")

    def upload(self, local_path, hdfs_path, overwrite=True):
        """로컬 파일을 HDFS에 업로드"""
        if not self.exists(hdfs_path) or overwrite:
            self.hdfs.upload(hdfs_path, local_path, overwrite=overwrite)
            print(f"  업로드 완료: {local_path} → {hdfs_path}")
        else:
            print(f"  이미 존재합니다 (overwrite=False): {hdfs_path}")

    # --- 관리 기능 ---
    def mkdir(self, path):
        """HDFS 디렉토리 생성"""
        self.hdfs.makedirs(path)
        print(f"  디렉토리 생성: {path}")

    def rename(self, src, dst):
        """파일/디렉토리 이름 변경"""
        self.hdfs.rename(src, dst)
        print(f"  이름 변경: {src} → {dst}")

    def delete(self, path, recursive=False):
        """파일 또는 디렉토리 삭제"""
        if not self.exists(path):
            print(f"  없는 경로: {path}")
            return
        self.hdfs.delete(path, recursive=recursive)
        print(f"  삭제 완료: {path}")


# ===== 사용 예시 =====
mgr = HdfsManager()
print()

# 1. 상세 목록 조회
print("[1] /dataset 상세 목록")
mgr.list_dir('/dataset', detail=True)
print()

In [ ]:
# 2. CSV → DataFrame 분석
print("[2] CSV 파일을 DataFrame으로 읽기 & 분석")
df = mgr.read_csv('/dataset/부산광역시방범용CCTV정보202512.csv', encoding='cp949')
print(df.head(3))
print(f"\n  컬럼: {list(df.columns)}")
print(f"  구군별 CCTV 수:\n{df['구군'].value_counts().head(5)}")
print()

In [ ]:
# 결측값 카운트
print("결측값 개수 (컬럼별):")
print(df.isnull().sum())
print(f"\n전체 결측값: {df.isnull().sum().sum()}개")
print(f"결측값 있는 컬럼: {df.isnull().any().sum()}개")

In [ ]:
# 3. 텍스트 직접 쓰기 → 이름 변경 → 삭제
print("[3] 텍스트 파일 쓰기")
mgr.write_text('/dataset/test_write.txt', "HDFS 쓰기 연습\nHello, HDFS!\n테스트 완료", encoding='utf-8')

print("\n[4] 파일 존재 확인")
print(f"  test_write.txt 존재: {mgr.exists('/dataset/test_write.txt')}")

print("\n[5] 파일 이름 변경")
mgr.rename('/dataset/test_write.txt', '/dataset/test_renamed.txt')

print("\n[6] 이름 변경 후 목록 확인")
mgr.list_dir('/dataset')

print("\n[7] 임시 파일 삭제")
mgr.delete('/dataset/test_renamed.txt')

print("\n[8] 삭제 후 목록 확인")
mgr.list_dir('/dataset')

In [ ]:
# 9. 디렉토리 생성 → 파일 업로드 → 디렉토리 전체 삭제
print("[9] 서브 디렉토리 생성 & 업로드 & 재귀 삭제")
mgr.mkdir('/dataset/temp_dir')
mgr.upload('./dataset/청소년정신건강20260604.csv', '/dataset/temp_dir/청소년정신건강.csv')

print("\n  temp_dir 내 목록:")
mgr.list_dir('/dataset/temp_dir')

print("\n  temp_dir 재귀 삭제")
mgr.delete('/dataset/temp_dir', recursive=True)
print(f"  temp_dir 존재: {mgr.exists('/dataset/temp_dir')}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

locations = df['location'].unique()
n = len(locations)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4.2),
                         subplot_kw={'projection': 'polar'})
axes = axes.flatten()

for i, loc in enumerate(locations):
    ax = axes[i]
    sub = df[df['location'] == loc].sort_values('year')

    years   = sub['year'].values
    percent = sub['percent'].values

    # 연도 → 각도 (0 ~ 2π, 균등 배치)
    n_years = len(years)
    theta = np.linspace(0, 2 * np.pi, n_years, endpoint=False)

    # 닫힌 다각형을 위해 첫 값을 끝에 추가
    theta_closed = np.append(theta, theta[0])
    r_closed     = np.append(percent, percent[0])

    # 색상: percent 크기에 따라 그라디언트
    colors = cm.RdYlGn(percent / percent.max())

    # 극좌표 막대 (bar)
    bar_width = 2 * np.pi / n_years * 0.85
    bars = ax.bar(theta, percent, width=bar_width, color=colors,
                  alpha=0.7, linewidth=0.3, edgecolor='gray')

    # 외곽선
    ax.plot(theta_closed, r_closed, color='steelblue', linewidth=1.2, alpha=0.8)

    # 연도 레이블: 5년 간격만 표시
    tick_idx = [j for j, y in enumerate(years) if y % 5 == 0]
    ax.set_xticks(theta[tick_idx])
    ax.set_xticklabels([str(years[j]) for j in tick_idx], fontsize=6.5)
    ax.set_yticklabels([])          # 반지름 숫자 숨김
    ax.set_ylim(0, percent.max() * 1.2)

    # 최대/최소 연도 강조 표시
    max_idx = np.argmax(percent)
    min_idx = np.argmin(percent)
    ax.annotate(f"{percent[max_idx]:.1f}%",
                xy=(theta[max_idx], percent[max_idx]),
                fontsize=6, color='darkgreen', ha='center')
    ax.annotate(f"{percent[min_idx]:.1f}%",
                xy=(theta[min_idx], percent[min_idx]),
                fontsize=6, color='crimson', ha='center')

    ax.set_title(loc, fontsize=8.5, pad=8, fontweight='bold')

# 빈 서브플롯 숨기기
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('지역별 연도별 비율 변화 (극좌표)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()